# Sign Language Recognition with Deep Learning
Recognize hand gestures from **images** and **video** and convert them to **text**.

| Part | Model | Input | Dataset |
|---|---|---|---|
| A | **CNN** | single image | ASL Alphabet (29 classes: A–Z, space, del, nothing) |
| B | **CNN-LSTM** | sequence of frames (video) | WLASL (word-level signs) |
| C | Inference | image / video / webcam → text | both |

**Why two models?** A static letter can be recognized from one frame, so a CNN is enough. Most words are *motion*, so you need a CNN to extract per-frame features and an LSTM to model how they change over time.

**Requirements:** `tensorflow>=2.12`, `opencv-python`, `kagglehub`, `scikit-learn`, `matplotlib`, `seaborn`. A GPU is strongly recommended (Colab / Kaggle free GPU works).

**Kaggle access:** `kagglehub` needs your Kaggle credentials (`~/.kaggle/kaggle.json` or `KAGGLE_USERNAME` / `KAGGLE_KEY` env vars).

In [ ]:
# %pip install -q tensorflow opencv-python kagglehub scikit-learn matplotlib seaborn
import os, json, glob, random, time
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## Part A: CNN on the ASL Alphabet (image → letter)

### A1. Download the dataset
[`grassknoted/asl-alphabet`](https://www.kaggle.com/datasets/grassknoted/asl-alphabet): about 87,000 images of 200×200 px, 29 classes.

In [ ]:
import kagglehub
asl_root = kagglehub.dataset_download("grassknoted/asl-alphabet")
print("Downloaded to:", asl_root)

# Folder layout differs slightly between versions, so locate the train folder automatically
train_dir = None
for p in Path(asl_root).rglob("asl_alphabet_train"):
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    if len(subdirs) >= 26:
        train_dir = p; break
    inner = p / "asl_alphabet_train"
    if inner.exists():
        train_dir = inner; break
print("Train dir:", train_dir)
print("Classes:", sorted(d.name for d in Path(train_dir).iterdir() if d.is_dir()))

### A2. Build `tf.data` pipelines (train / val / test)

In [ ]:
IMG_SIZE = 64        # 64x64 trains fast; raise to 96/128 for more accuracy
BATCH    = 64

train_ds = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="int")
val_full = keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="int")

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

# Split the held-out 20% in half: validation (during training) and test (final report)
n_val_batches = tf.data.experimental.cardinality(val_full).numpy()
val_ds  = val_full.take(n_val_batches // 2)
test_ds = val_full.skip(n_val_batches // 2)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)
print(NUM_CLASSES, "classes")

In [ ]:
# Peek at a few samples
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

### A3. Define the CNN
Augmentation uses rotation, zoom and shifts for robustness to camera position. **No horizontal flip**, because flipping turns a right hand into a left hand and changes the sign's appearance.

In [ ]:
def build_cnn(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    inp = layers.Input((img_size, img_size, 3))
    x = layers.Rescaling(1./255)(inp)                  # normalization lives inside the model
    x = layers.RandomRotation(0.08)(x)                 # augmentation (active only in training)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomContrast(0.2)(x)

    for filters in (32, 64, 128, 256):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out, name="asl_cnn")

cnn = build_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

### A4. Train

In [ ]:
EPOCHS = 20
cbs = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    callbacks.ModelCheckpoint("asl_cnn_best.keras", monitor="val_accuracy", save_best_only=True),
]
history = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs)

In [ ]:
def plot_history(h, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.history["accuracy"], label="train"); ax[0].plot(h.history["val_accuracy"], label="val")
    ax[0].set_title(f"{title} accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history["loss"], label="train"); ax[1].plot(h.history["val_loss"], label="val")
    ax[1].set_title(f"{title} loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.show()
plot_history(history, "CNN")

### A5. Evaluate on the test split

In [ ]:
test_loss, test_acc = cnn.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_true, y_pred = [], []
for x, y in test_ds:
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(cnn.predict(x, verbose=0), axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("CNN confusion matrix"); plt.show()

### A6. Predict from a single image → text
Real-world photos differ from this dataset (it was captured against a uniform background). If accuracy drops on your own images, crop tightly around the hand, and consider fine-tuning on your own samples.

In [ ]:
def predict_letter(img_bgr, model=cnn, top_k=3):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
    probs = model.predict(rgb[None].astype("float32"), verbose=0)[0]
    top = probs.argsort()[::-1][:top_k]
    return [(class_names[i], float(probs[i])) for i in top]

# Demo on a random training image
sample = random.choice(glob.glob(str(Path(train_dir) / "*" / "*.jpg")))
img = cv2.imread(sample)
print("True label:", Path(sample).parent.name)
print("Prediction:", predict_letter(img))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

---
## Part B: CNN-LSTM on video (WLASL: sign → word)

**Architecture:** every frame goes through a CNN (`TimeDistributed`) to produce a feature vector → the sequence of vectors goes into an LSTM that learns the motion → softmax over words.

```
video (T frames) → [CNN per frame] → (T, 1280) → LSTM → Dense → word
```

We use a frozen, pretrained **MobileNetV2** as the per-frame CNN. WLASL is small per class, so transfer learning matters a lot.

### B1. Get WLASL
Uses the processed Kaggle copy [`risangbaskoro/wlasl-processed`](https://www.kaggle.com/datasets/risangbaskoro/wlasl-processed) (videos + `WLASL_v0.3.json`). If you can't download it, the notebook falls back to a **pseudo-sequence sanity check** built from the ASL Alphabet, which only verifies the architecture, not real sign motion.

In [ ]:
NUM_WORDS  = 20     # use the top-N glosses by sample count (WLASL2000 is very hard; start small)
SEQ_LEN    = 16     # frames per clip
SEQ_SIZE   = 96     # frame resolution fed to MobileNetV2
USE_WLASL  = True

wlasl_json, video_dir = None, None
if USE_WLASL:
    try:
        wl_root = kagglehub.dataset_download("risangbaskoro/wlasl-processed")
        wlasl_json = next(Path(wl_root).rglob("WLASL_v0.3.json"), None)
        vids = [p for p in Path(wl_root).rglob("*") if p.is_dir() and len(list(p.glob("*.mp4"))) > 100]
        video_dir = vids[0] if vids else None
        print("JSON:", wlasl_json); print("Videos:", video_dir)
    except Exception as e:
        print("WLASL download failed:", e)
USE_WLASL = bool(wlasl_json and video_dir)
print("Using real WLASL:", USE_WLASL)

In [ ]:
def read_clip(path, n_frames=SEQ_LEN, size=SEQ_SIZE, bbox=None, start=1, end=-1):
    """Uniformly sample n_frames from a video, optionally crop to bbox, resize, return (T,H,W,3) uint8 RGB."""
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    s = max(start - 1, 0)
    e = total - 1 if end in (-1, None) else min(end - 1, total - 1)
    if e <= s: s, e = 0, total - 1
    idxs = np.linspace(s, e, n_frames).astype(int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, f = cap.read()
        if not ok:
            f = frames[-1] if frames else None
            if f is None: continue
            frames.append(f); continue
        if bbox is not None:
            h, w = f.shape[:2]
            x1, y1, x2, y2 = [int(v) for v in bbox]
            x1, y1, x2, y2 = max(x1,0), max(y1,0), min(x2,w), min(y2,h)
            if x2 > x1 and y2 > y1: f = f[y1:y2, x1:x2]
        f = cv2.cvtColor(cv2.resize(f, (size, size)), cv2.COLOR_BGR2RGB)
        frames.append(f)
    cap.release()
    if len(frames) == 0: return None
    while len(frames) < n_frames: frames.append(frames[-1])
    return np.stack(frames[:n_frames]).astype("uint8")


def load_wlasl(json_path, video_dir, num_words=NUM_WORDS):
    meta = json.load(open(json_path))
    avail = {p.stem for p in Path(video_dir).glob("*.mp4")}
    entries = []
    for e in meta:
        inst = [i for i in e["instances"] if i["video_id"] in avail]
        if inst: entries.append((e["gloss"], inst))
    entries.sort(key=lambda t: len(t[1]), reverse=True)
    entries = entries[:num_words]
    glosses = [g for g, _ in entries]
    data = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for label, (g, inst) in enumerate(entries):
        for i in inst:
            clip = read_clip(Path(video_dir) / f"{i['video_id']}.mp4",
                             bbox=i.get("bbox"), start=i.get("frame_start", 1), end=i.get("frame_end", -1))
            if clip is None: continue
            data[i["split"]][0].append(clip); data[i["split"]][1].append(label)
    out = {k: (np.array(v[0]), np.array(v[1])) for k, v in data.items()}
    return out, glosses

In [ ]:
if USE_WLASL:
    t0 = time.time()
    splits, seq_classes = load_wlasl(wlasl_json, video_dir)
    print(f"Loaded in {time.time()-t0:.0f}s")
else:
    # Sanity-check fallback: a "clip" = SEQ_LEN random images of the same ASL letter (no real motion!)
    print("Building pseudo-sequences from the ASL Alphabet (architecture test only).")
    seq_classes = [c for c in class_names if c not in ("del", "nothing", "space")][:NUM_WORDS]
    def make_split(n_per_class):
        X, y = [], []
        for li, c in enumerate(seq_classes):
            files = glob.glob(str(Path(train_dir) / c / "*.jpg"))
            for _ in range(n_per_class):
                pick = random.sample(files, SEQ_LEN)
                X.append(np.stack([cv2.cvtColor(cv2.resize(cv2.imread(f), (SEQ_SIZE, SEQ_SIZE)), cv2.COLOR_BGR2RGB) for f in pick]))
                y.append(li)
        return np.array(X, dtype="uint8"), np.array(y)
    splits = {"train": make_split(60), "val": make_split(10), "test": make_split(10)}

for k, (X, y) in splits.items(): print(k, X.shape, y.shape)
print("Classes:", seq_classes)

In [ ]:
# Show frames of one clip
X_tr, y_tr = splits["train"]
clip = X_tr[0]
plt.figure(figsize=(16, 2.2))
for i in range(0, SEQ_LEN, 2):
    plt.subplot(1, SEQ_LEN // 2, i // 2 + 1); plt.imshow(clip[i]); plt.axis("off"); plt.title(f"t={i}")
plt.suptitle(f"Sample clip: {seq_classes[y_tr[0]]}"); plt.show()

### B2. Build the CNN-LSTM
Phase 1 trains only the LSTM head on frozen MobileNetV2 features. Phase 2 (optional) unfreezes the top of the CNN at a low learning rate.

In [ ]:
def build_cnn_lstm(num_classes, seq_len=SEQ_LEN, size=SEQ_SIZE, lstm_units=128):
    base = keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                                          input_shape=(size, size, 3), pooling="avg")
    base.trainable = False

    inp = layers.Input((seq_len, size, size, 3))
    x = layers.TimeDistributed(layers.Rescaling(1./127.5, offset=-1))(inp)   # MobileNetV2 expects [-1, 1]
    x = layers.TimeDistributed(base, name="frame_cnn")(x)                    # (B, T, 1280)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))(x)
    x = layers.LSTM(lstm_units)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out, name="cnn_lstm"), base

cnn_lstm, base_cnn = build_cnn_lstm(len(seq_classes))
cnn_lstm.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_lstm.summary()

In [ ]:
def augment_clip(clip, label):
    """Same random shift/brightness for all frames of one clip (keeps temporal consistency)."""
    clip = tf.cast(clip, tf.float32)
    clip = tf.image.random_brightness(clip, 20.0)
    shift = tf.random.uniform([2], -8, 8, dtype=tf.int32)
    clip = tf.roll(clip, shift=shift, axis=[1, 2])
    return tf.clip_by_value(clip, 0, 255), label

def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training: ds = ds.shuffle(len(X), seed=SEED).map(augment_clip, num_parallel_calls=AUTOTUNE)
    else:        ds = ds.map(lambda a, b: (tf.cast(a, tf.float32), b))
    return ds.batch(8).prefetch(AUTOTUNE)

seq_train = make_ds(*splits["train"], True)
seq_val   = make_ds(*splits["val"],   False)
seq_test  = make_ds(*splits["test"],  False)

### B3. Train (phase 1: frozen CNN, then phase 2: fine-tune)

In [ ]:
cbs = [callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
       callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]
h1 = cnn_lstm.fit(seq_train, validation_data=seq_val, epochs=30, callbacks=cbs)

In [ ]:
# Phase 2 (optional): unfreeze the last ~30 layers of MobileNetV2, train with a small LR
base_cnn.trainable = True
for layer in base_cnn.layers[:-30]:
    layer.trainable = False
cnn_lstm.compile(optimizer=keras.optimizers.Adam(1e-5),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = cnn_lstm.fit(seq_train, validation_data=seq_val, epochs=15, callbacks=cbs)
plot_history(h1, "CNN-LSTM (phase 1)"); plot_history(h2, "CNN-LSTM (phase 2)")
cnn_lstm.save("cnn_lstm_best.keras")

### B4. Evaluate

In [ ]:
loss, acc = cnn_lstm.evaluate(seq_test, verbose=0)
print(f"CNN-LSTM test accuracy: {acc:.4f}")

yt, yp = [], []
for x, y in seq_test:
    yt.extend(y.numpy()); yp.extend(np.argmax(cnn_lstm.predict(x, verbose=0), axis=1))
print(classification_report(yt, yp, labels=range(len(seq_classes)), target_names=seq_classes, digits=3, zero_division=0))

plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(yt, yp, labels=range(len(seq_classes))), cmap="Blues",
            xticklabels=seq_classes, yticklabels=seq_classes)
plt.title("CNN-LSTM confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

---
## Part C: Inference → text

### C1. Video file → word (CNN-LSTM)

In [ ]:
def predict_video(path, model=cnn_lstm, classes=seq_classes, top_k=3):
    clip = read_clip(path)
    if clip is None: raise ValueError("Could not read video")
    probs = model.predict(clip[None].astype("float32"), verbose=0)[0]
    top = probs.argsort()[::-1][:top_k]
    return [(classes[i], float(probs[i])) for i in top]

# Demo on a test sample (replace with your own .mp4 when using real WLASL)
if USE_WLASL:
    p = random.choice(list(Path(video_dir).glob("*.mp4")))
    print(p.name, "->", predict_video(p))

### C2. Live webcam → text (CNN, letters)
Run this **locally** (it opens a window, so it won't work on Colab). Hold a sign inside the green box; a letter is added to the sentence once it stays stable for ~15 frames at high confidence.
Keys: **q** quit · **c** clear · **backspace** delete last character.

In [ ]:
def run_webcam(model=cnn, classes=class_names, img_size=IMG_SIZE,
               conf_thr=0.90, stable_frames=15, box=300):
    cap = cv2.VideoCapture(0)
    text, last, count = "", None, 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        h, w = frame.shape[:2]
        x1, y1 = w - box - 40, 40
        roi = frame[y1:y1 + box, x1:x1 + box]
        rgb = cv2.resize(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB), (img_size, img_size))
        probs = model.predict(rgb[None].astype("float32"), verbose=0)[0]
        idx = int(probs.argmax()); label, conf = classes[idx], float(probs[idx])

        if conf >= conf_thr and label == last: count += 1
        else: count = 0
        last = label

        if count == stable_frames:
            if label == "space": text += " "
            elif label == "del": text = text[:-1]
            elif label != "nothing": text += label
            count = -stable_frames          # cooldown to avoid repeats

        cv2.rectangle(frame, (x1, y1), (x1 + box, y1 + box), (0, 255, 0), 2)
        cv2.putText(frame, f"{label} ({conf:.0%})", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.rectangle(frame, (0, h - 50), (w, h), (0, 0, 0), -1)
        cv2.putText(frame, text[-40:], (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        cv2.imshow("Sign language to text", frame)

        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == ord("c"): text = ""
        if k == 8:        text = text[:-1]
    cap.release(); cv2.destroyAllWindows()
    return text

# final_text = run_webcam()      # uncomment to run locally
# print(final_text)

### C3. Webcam for words (CNN-LSTM, sliding window)
Keep a rolling buffer of the last `SEQ_LEN` frames (sampled every few frames) and classify it whenever the buffer is full.

In [ ]:
from collections import deque

def run_webcam_words(model=cnn_lstm, classes=seq_classes, conf_thr=0.6, stride=3):
    cap = cv2.VideoCapture(0)
    buf, words, n = deque(maxlen=SEQ_LEN), [], 0
    shown, shown_conf = "", 0.0
    while True:
        ok, frame = cap.read()
        if not ok: break
        n += 1
        if n % stride == 0:
            buf.append(cv2.cvtColor(cv2.resize(frame, (SEQ_SIZE, SEQ_SIZE)), cv2.COLOR_BGR2RGB))
        if len(buf) == SEQ_LEN and n % (stride * 8) == 0:
            probs = model.predict(np.array(buf)[None].astype("float32"), verbose=0)[0]
            i = int(probs.argmax())
            shown, shown_conf = classes[i], float(probs[i])
            if shown_conf >= conf_thr and (not words or words[-1] != shown):
                words.append(shown)
        cv2.putText(frame, f"{shown} ({shown_conf:.0%})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.putText(frame, " ".join(words[-6:]), (10, frame.shape[0] - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        cv2.imshow("Sign to words", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"): break
    cap.release(); cv2.destroyAllWindows()
    return " ".join(words)

# print(run_webcam_words())     # uncomment to run locally

## Notes and next steps

**Honest expectations**
- ASL Alphabet CNN: typically 95–99%+ on the held-out split, but this overstates real-world accuracy because images share the same background, lighting and signer. Test on your own camera.
- WLASL is a hard benchmark: it has few videos per word, many signers and noisy labels. Expect modest accuracy on 20 words with a small CNN-LSTM and much lower on WLASL-100/300/2000. State-of-the-art uses pose/transformer models.

**Ways to improve**
1. **MediaPipe Hands/Holistic landmarks** instead of raw pixels: much more robust to background, lighting and skin tone. Feed 21×3 landmarks per frame (×2 hands) into the LSTM.
2. Fine-tune on your own webcam data (a few hundred frames per letter helps a lot).
3. Use a 3D CNN (I3D) or a Video Transformer for the temporal part.
4. Add a language model or dictionary to smooth letter streams into words.
5. Save models with `model.save()` and convert to TFLite for mobile or edge deployment.